In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [5]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

DATA_DIR = "/kaggle/input/datasets/serenaraju/yawn-eye-dataset-new/dataset_new"
IMG_SIZE = (160, 160)
BATCH = 32

train_gen = ImageDataGenerator(
    rescale=1./255, validation_split=0.2,
    rotation_range=10, zoom_range=0.15,
    width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True
)
train_data = train_gen.flow_from_directory(f"{DATA_DIR}/train", target_size=IMG_SIZE, batch_size=BATCH, subset="training")
val_data = train_gen.flow_from_directory(f"{DATA_DIR}/train", target_size=IMG_SIZE, batch_size=BATCH, subset="validation")

base = MobileNetV2(input_shape=(160,160,3), include_top=False, weights="imagenet")
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dropout(0.3)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(train_data.num_classes, activation="softmax")(x)
model = Model(base.input, output)

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

early_stop = EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)
model.fit(train_data, validation_data=val_data, epochs=20, callbacks=[early_stop])

model.save("drowsiness_model.h5")
print(train_data.class_indices)

# Evaluate on the real held-out test set
test_gen = ImageDataGenerator(rescale=1./255)
test_data = test_gen.flow_from_directory(f"{DATA_DIR}/test", target_size=IMG_SIZE, batch_size=BATCH, shuffle=False)
loss, acc = model.evaluate(test_data)
print("Test accuracy:", acc)

Found 1975 images belonging to 4 classes.
Found 492 images belonging to 4 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 56s 685ms/step - accuracy: 0.7423 - loss: 0.5676 - val_accuracy: 0.8171 - val_loss: 0.4178
Epoch 2/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 20s 328ms/step - accuracy: 0.8314 - loss: 0.3548 - val_accuracy: 0.7744 - val_loss: 0.3562
Epoch 3/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 20s 318ms/step - accuracy: 0.8354 - loss: 0.3105 - val_accuracy: 0.7561 - val_loss: 0.4109
Epoch 4/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 19s 312ms/step - accuracy: 0.8435 - loss: 0.3016 - val_accuracy: 0.7825 - val_loss: 0.3707
Epoch 5/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 21s 337ms/step - accuracy: 0.8658 - loss: 0.2722 - val_accuracy: 0.7337 - val_loss: 0.4313


{'Closed': 0, 'Open': 1, 'no_yawn': 2, 'yawn': 3}
Found 433 images belonging to 4 classes.
13/14 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.9443 - loss: 0.1640

2026-08-07 06:26:46.328174: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-07 06:26:46.465871: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-07 06:26:46.602143: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


14/14 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.8106 - loss: 0.4404 
Test accuracy: 0.8106235861778259
